<a href="https://colab.research.google.com/github/apmontesp/NLP_encoder-decoder-attention_summarization/blob/main/CNN_DailyMail_EncoderDecoder_Attention.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Resumen Automático de Noticias con Encoder-Decoder y Atención

**Taller de Procesamiento de Lenguaje Natural**

- Tarea: Resumen abstractivo de texto (*abstractive text summarization*).
- Dataset: `cnn_dailymail` v3.0.0 (Hugging Face Datasets).
- Arquitectura: Encoder BiLSTM + Atención de Bahdanau + Decoder LSTM.
- Embeddings: GloVe 6B 100d (Pennington et al., 2014).

---


## 1. Contexto del problema

### 1.1 Descripción del dataset CNN/DailyMail

El corpus **CNN/DailyMail** fue introducido por Hermann et al. (2015) para la tarea de
*reading comprehension* y posteriormente adaptado por Nallapati et al. (2016) y
See et al. (2017) como benchmark estándar de resumen abstractivo. Contiene
aproximadamente **287.000 pares** de entrenamiento, **13.300** de validación y
**11.500** de prueba, donde cada par consta de un artículo periodístico y los
*highlights* asociados que actúan como resumen de referencia.

### 1.2 Problema que resuelve

Un modelo entrenado sobre este corpus puede aprender a generar **resúmenes
abstractivos** de noticias. A diferencia del resumen extractivo —que selecciona
oraciones literales del documento—, la formulación abstractiva exige al modelo:

1. Comprender el contenido global del artículo.
2. Identificar la información saliente.
3. Generar texto nuevo que sintetice esa información en pocas oraciones.

| Aspecto | Artículo (entrada) | Resumen (salida) |
|---|---|---|
| Longitud media | ~780 palabras | ~55 palabras |
| Tipo de texto | Noticia periodística | Texto abstractivo conciso |
| Ratio de compresión | — | ≈ 14× |

### 1.3 Aplicaciones

Los modelos entrenados sobre CNN/DailyMail son aplicables a la generación de
resúmenes de noticias, informes técnicos, documentos legales o financieros, así
como a la indexación automática de grandes corpus documentales y a sistemas de
asistencia a la lectura.


## 2. Instalación de dependencias

In [ ]:
# Instalación de librerías requeridas
!pip install -q datasets transformers rouge-score nltk torch torchtext evaluate accelerate sentencepiece


In [ ]:
import os
import re
import math
import time
import random
import urllib.request
import zipfile
import pickle
from collections import Counter

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence

import nltk
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)

from datasets import load_dataset
from rouge_score import rouge_scorer

# Reproducibilidad
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True

# Dispositivo de cómputo
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'[✓] Dispositivo activo: {device}')
if torch.cuda.is_available():
    print(f'    GPU      : {torch.cuda.get_device_name(0)}')
    print(f'    Memoria  : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')


## 3. Carga del dataset desde Hugging Face

El dataset se carga directamente desde el *hub* oficial de Hugging Face Datasets
mediante la función `load_dataset`. La versión utilizada es la 3.0.0, que
corresponde al *split* canónico empleado en la literatura.


In [ ]:
# Carga del dataset CNN/DailyMail desde Hugging Face
print('Cargando CNN/DailyMail v3.0.0 ...')
dataset = load_dataset('cnn_dailymail', '3.0.0')
print(f'[✓] Dataset cargado correctamente.')
print(dataset)


In [ ]:
# Particiones
train_data = dataset['train']
val_data   = dataset['validation']
test_data  = dataset['test']

print('Tamaño de las particiones:')
print(f'    Entrenamiento : {len(train_data):,} ejemplos')
print(f'    Validación    : {len(val_data):,} ejemplos')
print(f'    Prueba        : {len(test_data):,} ejemplos')

sample = train_data[0]
print('\nEjemplo de artículo (primeros 500 caracteres):')
print('-' * 60)
print(sample['article'][:500])
print('\nResumen de referencia (highlights):')
print('-' * 60)
print(sample['highlights'])


In [ ]:
# Análisis de longitudes en una submuestra del conjunto de entrenamiento
sample_size = 5000
indices = random.sample(range(len(train_data)), sample_size)
article_lens = [len(train_data[i]['article'].split()) for i in indices]
summary_lens = [len(train_data[i]['highlights'].split()) for i in indices]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Distribución de longitudes — CNN/DailyMail',
             fontsize=14, fontweight='bold')

axes[0].hist(article_lens, bins=50, color='steelblue', alpha=0.85, edgecolor='white')
axes[0].axvline(np.mean(article_lens), color='red', linestyle='--',
                label=f'Media = {np.mean(article_lens):.0f}')
axes[0].set_title('Artículos (palabras)')
axes[0].set_xlabel('Número de palabras')
axes[0].set_ylabel('Frecuencia')
axes[0].legend()

axes[1].hist(summary_lens, bins=50, color='coral', alpha=0.85, edgecolor='white')
axes[1].axvline(np.mean(summary_lens), color='navy', linestyle='--',
                label=f'Media = {np.mean(summary_lens):.0f}')
axes[1].set_title('Resúmenes (palabras)')
axes[1].set_xlabel('Número de palabras')
axes[1].legend()

plt.tight_layout()
plt.savefig('length_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print('Estadísticas descriptivas:')
print(f'    Artículos  - media: {np.mean(article_lens):.0f} | mediana: {np.median(article_lens):.0f} | max: {max(article_lens)}')
print(f'    Resúmenes  - media: {np.mean(summary_lens):.0f} | mediana: {np.median(summary_lens):.0f} | max: {max(summary_lens)}')
print(f'    Ratio medio de compresión: {np.mean(article_lens)/np.mean(summary_lens):.1f}x')


## 4. Preprocesamiento y construcción de vocabulario

Se construye un vocabulario *word-level* sobre la submuestra de entrenamiento.
Los embeddings preentrenados de GloVe 6B 100d se asignan a las palabras del
vocabulario; las palabras fuera de cobertura se inicializan con una distribución
uniforme y se permiten ajustes durante el entrenamiento (*fine-tuning*).


In [ ]:
# ────────────────────────────────────────────────────────────────────────
# Modo rápido para verificación del pipeline
# ────────────────────────────────────────────────────────────────────────
# FAST_MODE = True  -> entrena con un subconjunto pequeño y modelo reducido
#                      (5-10 min en GPU). Útil para comprobar que todas las
#                      celdas se ejecutan sin error.
# FAST_MODE = False -> configuración académica completa (mejores resultados,
#                      varias horas de entrenamiento).
FAST_MODE = True

if FAST_MODE:
    # Hiperparámetros reducidos para verificación rápida
    MAX_ARTICLE_LEN = 200       # vs 400
    MAX_SUMMARY_LEN = 60        # vs 100
    VOCAB_SIZE      = 10000     # vs 30000
    EMBED_DIM       = 100       # GloVe 6B.100d (no cambia)
    HIDDEN_DIM      = 128       # vs 256
    N_LAYERS        = 1         # vs 2
    DROPOUT         = 0.2
    BATCH_SIZE      = 16        # vs 32
    LEARNING_RATE   = 1e-3
    N_EPOCHS        = 1         # vs 5
    CLIP            = 1.0
    TRAIN_SAMPLES   = 2000      # vs 50000
    VAL_SAMPLES     = 400       # vs 5000
    ROUGE_SAMPLES   = 50        # vs 200 (usado en la sección de evaluación)
    print('[✓] FAST_MODE activado — configuración para verificación rápida.')
else:
    # Configuración académica para entrenamiento completo
    MAX_ARTICLE_LEN = 400
    MAX_SUMMARY_LEN = 100
    VOCAB_SIZE      = 30000
    EMBED_DIM       = 100
    HIDDEN_DIM      = 256
    N_LAYERS        = 2
    DROPOUT         = 0.3
    BATCH_SIZE      = 32
    LEARNING_RATE   = 1e-3
    N_EPOCHS        = 5
    CLIP            = 1.0
    TRAIN_SAMPLES   = 50000
    VAL_SAMPLES     = 5000
    ROUGE_SAMPLES   = 200
    print('[✓] Modo completo — configuración académica.')

# Tokens especiales
PAD_TOKEN, UNK_TOKEN, SOS_TOKEN, EOS_TOKEN = '<pad>', '<unk>', '<sos>', '<eos>'

print(f'[✓] Hiperparámetros configurados.')
for k, v in [
    ('MAX_ARTICLE_LEN', MAX_ARTICLE_LEN),
    ('MAX_SUMMARY_LEN', MAX_SUMMARY_LEN),
    ('VOCAB_SIZE', VOCAB_SIZE),
    ('EMBED_DIM', EMBED_DIM),
    ('HIDDEN_DIM', HIDDEN_DIM),
    ('N_LAYERS', N_LAYERS),
    ('DROPOUT', DROPOUT),
    ('BATCH_SIZE', BATCH_SIZE),
    ('LEARNING_RATE', LEARNING_RATE),
    ('N_EPOCHS', N_EPOCHS),
    ('TRAIN_SAMPLES', TRAIN_SAMPLES),
    ('VAL_SAMPLES', VAL_SAMPLES),
]:
    print(f'    {k:<18} : {v}')


In [ ]:
def simple_tokenize(text):
    """Normalización y tokenización: minúsculas y separación por espacios."""
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s']", ' ', text)
    return text.split()


class Vocabulary:
    """Vocabulario word-level con tokens especiales."""

    def __init__(self, max_size=None):
        self.max_size = max_size
        self.word2idx = {}
        self.idx2word = {}
        self.word_freq = Counter()
        for i, tok in enumerate([PAD_TOKEN, UNK_TOKEN, SOS_TOKEN, EOS_TOKEN]):
            self.word2idx[tok] = i
            self.idx2word[i] = tok
        self.pad_idx, self.unk_idx, self.sos_idx, self.eos_idx = 0, 1, 2, 3

    def build(self, texts):
        for text in texts:
            self.word_freq.update(simple_tokenize(text))
        cap = (self.max_size - 4) if self.max_size else None
        for word, _ in self.word_freq.most_common(cap):
            idx = len(self.word2idx)
            self.word2idx[word] = idx
            self.idx2word[idx] = word

    def encode(self, text, max_len=None, add_eos=False, add_sos=False):
        tokens = simple_tokenize(text)
        if max_len:
            tokens = tokens[:max_len]
        ids = [self.word2idx.get(t, self.unk_idx) for t in tokens]
        if add_sos:
            ids = [self.sos_idx] + ids
        if add_eos:
            ids = ids + [self.eos_idx]
        return ids

    def decode(self, ids, skip_special=True):
        special = {self.pad_idx, self.sos_idx, self.eos_idx}
        words = []
        for i in ids:
            if i == self.eos_idx:
                break
            if skip_special and i in special:
                continue
            words.append(self.idx2word.get(i, UNK_TOKEN))
        return ' '.join(words)

    def __len__(self):
        return len(self.word2idx)


# Construcción del vocabulario
print('Construyendo vocabulario (puede tomar 1-2 minutos) ...')
build_samples = TRAIN_SAMPLES or len(train_data)
all_texts = (
    [train_data[i]['article'] for i in range(build_samples)]
    + [train_data[i]['highlights'] for i in range(build_samples)]
)
vocab = Vocabulary(max_size=VOCAB_SIZE)
vocab.build(all_texts)
print(f'[✓] Vocabulario construido con {len(vocab):,} tokens.')
print('Top 20 palabras más frecuentes:')
print([w for w, _ in vocab.word_freq.most_common(20)])


In [ ]:
# Carga de embeddings preentrenados GloVe 6B 100d
GLOVE_PATH = 'glove.6B.100d.txt'

if not os.path.exists(GLOVE_PATH):
    print('Descargando GloVe 6B (~862 MB) ...')
    url = 'https://nlp.stanford.edu/data/glove.6B.zip'
    urllib.request.urlretrieve(url, 'glove.6B.zip')
    with zipfile.ZipFile('glove.6B.zip', 'r') as zf:
        zf.extract('glove.6B.100d.txt')
    print(f'[✓] Descarga completada.')
else:
    print(f'[✓] GloVe ya disponible localmente.')


def load_glove_embeddings(path, vocab, embed_dim):
    """Construye la matriz de embeddings alineada con el vocabulario."""
    print(f'Cargando GloVe ({embed_dim}d) ...')
    glove = {}
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.split()
            word = parts[0]
            if word in vocab.word2idx:
                glove[word] = np.array(parts[1:], dtype=np.float32)

    emb_matrix = np.random.uniform(
        -0.1, 0.1, (len(vocab), embed_dim)
    ).astype(np.float32)
    emb_matrix[vocab.pad_idx] = np.zeros(embed_dim)

    hits = 0
    for word, idx in vocab.word2idx.items():
        if word in glove:
            emb_matrix[idx] = glove[word]
            hits += 1

    coverage = hits / len(vocab) * 100
    print(f'[✓] Embeddings cargados: {hits:,}/{len(vocab):,} '
          f'({coverage:.1f}% cobertura).')
    return torch.FloatTensor(emb_matrix)


embedding_matrix = load_glove_embeddings(GLOVE_PATH, vocab, EMBED_DIM)


In [ ]:
class SummarizationDataset(Dataset):
    """Dataset para summarización: tensores fuente y objetivo."""

    def __init__(self, hf_dataset, vocab, max_article_len, max_summary_len, n_samples=None):
        self.vocab = vocab
        self.max_article_len = max_article_len
        self.max_summary_len = max_summary_len
        n = n_samples or len(hf_dataset)
        self.data = [hf_dataset[i] for i in range(n)]

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        src = self.vocab.encode(item['article'], max_len=self.max_article_len)
        trg = self.vocab.encode(
            item['highlights'],
            max_len=self.max_summary_len - 2,
            add_sos=True, add_eos=True,
        )
        return torch.LongTensor(src), torch.LongTensor(trg)


def collate_fn(batch):
    """Padding dinámico para batches de longitud variable."""
    srcs, trgs = zip(*batch)
    src_lens = [len(s) for s in srcs]
    src_padded = torch.zeros(len(srcs), max(src_lens), dtype=torch.long)
    trg_padded = torch.zeros(len(trgs), max(len(t) for t in trgs), dtype=torch.long)
    for i, (s, t) in enumerate(zip(srcs, trgs)):
        src_padded[i, :len(s)] = s
        trg_padded[i, :len(t)] = t
    return src_padded, torch.LongTensor(src_lens), trg_padded


print('Construyendo datasets y dataloaders ...')
train_dataset = SummarizationDataset(train_data, vocab, MAX_ARTICLE_LEN, MAX_SUMMARY_LEN, TRAIN_SAMPLES)
val_dataset   = SummarizationDataset(val_data,   vocab, MAX_ARTICLE_LEN, MAX_SUMMARY_LEN, VAL_SAMPLES)
test_dataset  = SummarizationDataset(test_data,  vocab, MAX_ARTICLE_LEN, MAX_SUMMARY_LEN, 1000)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          collate_fn=collate_fn, num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False,
                          collate_fn=collate_fn, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False,
                          collate_fn=collate_fn, num_workers=2, pin_memory=True)

print(f'[✓] DataLoaders creados:')
print(f'    Entrenamiento : {len(train_loader)} batches ({len(train_dataset):,} ejemplos)')
print(f'    Validación    : {len(val_loader)} batches ({len(val_dataset):,} ejemplos)')
print(f'    Prueba        : {len(test_loader)} batches ({len(test_dataset):,} ejemplos)')


## 5. Arquitectura: Encoder-Decoder con atención de Bahdanau

Se implementa la arquitectura *attention-based sequence-to-sequence* propuesta
por Bahdanau et al. (2015). El modelo consta de tres componentes:

```
Artículo --> [Encoder BiLSTM] --> {h_1, h_2, ..., h_T}
                                          |
                                  [Atención de Bahdanau] <-- s_t (Decoder)
                                          |
                                Vector de contexto c_t
                                          |
                              [Decoder LSTM] --> [Linear] --> distribución sobre vocabulario
```

| Componente | Descripción |
|---|---|
| Encoder | LSTM bidireccional, 2 capas, dimensión oculta 256 |
| Embeddings | GloVe 6B 100d, ajustables durante el entrenamiento |
| Atención | Aditiva (Bahdanau), proyección a `attn_dim = hidden_dim` |
| Decoder | LSTM unidireccional con concatenación `[embed; contexto]` |
| Salida | Capa lineal sobre el vocabulario |


In [ ]:
class Encoder(nn.Module):
    """Encoder LSTM bidireccional con embeddings GloVe."""

    def __init__(self, vocab_size, embed_dim, hidden_dim, n_layers, dropout, embedding_matrix):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.n_layers = n_layers

        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.embedding.weight = nn.Parameter(embedding_matrix)
        self.embedding.weight.requires_grad = True  # fine-tune

        self.rnn = nn.LSTM(
            embed_dim, hidden_dim, n_layers,
            batch_first=True, bidirectional=True,
            dropout=dropout if n_layers > 1 else 0,
        )
        self.dropout = nn.Dropout(dropout)
        self.fc_hidden = nn.Linear(hidden_dim * 2, hidden_dim)
        self.fc_cell   = nn.Linear(hidden_dim * 2, hidden_dim)

    def forward(self, src, src_lens):
        embedded = self.dropout(self.embedding(src))
        packed = pack_padded_sequence(
            embedded, src_lens.cpu(), batch_first=True, enforce_sorted=False
        )
        outputs, (hidden, cell) = self.rnn(packed)
        outputs, _ = pad_packed_sequence(outputs, batch_first=True)
        hidden = self._combine_bidir(hidden)
        cell   = self._combine_bidir(cell)
        return outputs, hidden, cell

    def _combine_bidir(self, state):
        """(2*layers, batch, H) --> (layers, batch, H)"""
        batch_size = state.shape[1]
        state = state.view(self.n_layers, 2, batch_size, self.hidden_dim)
        combined = torch.cat([state[:, 0], state[:, 1]], dim=2)
        projected = [
            torch.tanh(self.fc_hidden(combined[i])) for i in range(self.n_layers)
        ]
        return torch.stack(projected)


print(f'[✓] Encoder definido.')


In [ ]:
class BahdanauAttention(nn.Module):
    """Atención aditiva de Bahdanau et al. (2015).

    score(s_t, h_i) = v^T tanh(W_enc h_i + W_dec s_t)
    alpha_i        = softmax(score)
    contexto       = sum_i alpha_i h_i
    """

    def __init__(self, enc_hidden_dim, dec_hidden_dim, attn_dim=None):
        super().__init__()
        attn_dim = attn_dim or dec_hidden_dim
        self.attn_enc = nn.Linear(enc_hidden_dim * 2, attn_dim, bias=False)
        self.attn_dec = nn.Linear(dec_hidden_dim, attn_dim, bias=False)
        self.v = nn.Linear(attn_dim, 1, bias=False)

    def forward(self, encoder_outputs, decoder_hidden, src_mask=None):
        enc_proj = self.attn_enc(encoder_outputs)
        dec_proj = self.attn_dec(decoder_hidden).unsqueeze(1)
        energy = torch.tanh(enc_proj + dec_proj)
        scores = self.v(energy).squeeze(2)
        if src_mask is not None:
            scores = scores.masked_fill(src_mask == 0, float('-inf'))
        attn_weights = F.softmax(scores, dim=1)
        context = torch.bmm(attn_weights.unsqueeze(1), encoder_outputs).squeeze(1)
        return context, attn_weights


print(f'[✓] Mecanismo de atención de Bahdanau definido.')


In [ ]:
class Decoder(nn.Module):
    """Decoder LSTM con atención sobre los estados del encoder."""

    def __init__(self, vocab_size, embed_dim, enc_hidden_dim, dec_hidden_dim,
                 n_layers, dropout, embedding_matrix, attention):
        super().__init__()
        self.vocab_size = vocab_size
        self.attention  = attention

        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.embedding.weight = nn.Parameter(embedding_matrix)
        self.embedding.weight.requires_grad = True

        self.rnn = nn.LSTM(
            embed_dim + enc_hidden_dim * 2,
            dec_hidden_dim, n_layers,
            batch_first=True,
            dropout=dropout if n_layers > 1 else 0,
        )
        self.dropout = nn.Dropout(dropout)
        self.fc_out = nn.Linear(
            dec_hidden_dim + enc_hidden_dim * 2 + embed_dim, vocab_size
        )

    def forward(self, trg_token, hidden, cell, encoder_outputs, src_mask=None):
        trg_token = trg_token.unsqueeze(1)
        embedded  = self.dropout(self.embedding(trg_token))
        context, attn_weights = self.attention(encoder_outputs, hidden[-1], src_mask)
        context_exp = context.unsqueeze(1)
        rnn_input = torch.cat([embedded, context_exp], dim=2)
        output, (hidden, cell) = self.rnn(rnn_input, (hidden, cell))
        output  = output.squeeze(1)
        context = context_exp.squeeze(1)
        embedded = embedded.squeeze(1)
        prediction = self.fc_out(torch.cat([output, context, embedded], dim=1))
        return prediction, hidden, cell, attn_weights


print(f'[✓] Decoder definido.')


In [ ]:
class Seq2Seq(nn.Module):
    """Modelo Encoder-Decoder con teacher forcing."""

    def __init__(self, encoder, decoder, device, pad_idx=0):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device  = device
        self.pad_idx = pad_idx

    def create_mask(self, src):
        return (src != self.pad_idx)

    def forward(self, src, src_lens, trg, teacher_forcing_ratio=0.5):
        batch_size = src.shape[0]
        trg_len    = trg.shape[1]

        encoder_outputs, hidden, cell = self.encoder(src, src_lens)
        src_mask = self.create_mask(src)

        outputs   = torch.zeros(batch_size, trg_len, self.decoder.vocab_size).to(self.device)
        all_attn  = torch.zeros(batch_size, trg_len, src.shape[1]).to(self.device)

        input_token = trg[:, 0]  # <sos>
        for t in range(1, trg_len):
            output, hidden, cell, attn_w = self.decoder(
                input_token, hidden, cell, encoder_outputs, src_mask
            )
            outputs[:, t]  = output
            all_attn[:, t] = attn_w
            use_teacher = random.random() < teacher_forcing_ratio
            top1 = output.argmax(1)
            input_token = trg[:, t] if use_teacher else top1
        return outputs, all_attn


# Instanciación del modelo
attn    = BahdanauAttention(HIDDEN_DIM, HIDDEN_DIM)
encoder = Encoder(len(vocab), EMBED_DIM, HIDDEN_DIM, N_LAYERS, DROPOUT, embedding_matrix)
decoder = Decoder(len(vocab), EMBED_DIM, HIDDEN_DIM, HIDDEN_DIM, N_LAYERS, DROPOUT,
                  embedding_matrix, attn)
model   = Seq2Seq(encoder, decoder, device).to(device)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'[✓] Modelo instanciado.')
print(f'    Parámetros totales      : {total_params:,}')
print(f'    Parámetros entrenables  : {trainable_params:,}')
print('\nArquitectura:')
print(model)


## 6. Entrenamiento

Se utiliza el optimizador Adam con regularización L2 y un *scheduler*
`ReduceLROnPlateau` que reduce la tasa de aprendizaje cuando la pérdida de
validación se estanca. La función de costo es la entropía cruzada con
*ignore_index* sobre los tokens de relleno. Durante el entrenamiento se aplica
*gradient clipping* y un programa decreciente de *teacher forcing*.


In [ ]:
def init_weights(m):
    """Inicialización Xavier para parámetros que no son embeddings."""
    if hasattr(m, 'weight') and m.weight.dim() > 1:
        nn.init.xavier_uniform_(m.weight.data)


for _, module in model.named_modules():
    if not isinstance(module, nn.Embedding):
        module.apply(init_weights)

optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, patience=2, factor=0.5
)
criterion = nn.CrossEntropyLoss(ignore_index=vocab.pad_idx)

print(f'[✓] Optimizador y criterio configurados.')
print(f'    Optimizer  : Adam (lr={LEARNING_RATE}, weight_decay=1e-5)')
print(f'    Loss       : CrossEntropyLoss (ignore_index={vocab.pad_idx})')
print(f'    Scheduler  : ReduceLROnPlateau (patience=2, factor=0.5)')


In [ ]:
def train_epoch(model, loader, optimizer, criterion, clip, device, teacher_forcing_ratio=0.5):
    model.train()
    epoch_loss = 0.0
    for i, (src, src_lens, trg) in enumerate(loader):
        src, trg = src.to(device), trg.to(device)
        optimizer.zero_grad()
        outputs, _ = model(src, src_lens, trg, teacher_forcing_ratio)
        outputs = outputs[:, 1:].reshape(-1, outputs.shape[-1])
        trg     = trg[:, 1:].reshape(-1)
        loss = criterion(outputs, trg)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
        optimizer.step()
        epoch_loss += loss.item()
        if (i + 1) % 100 == 0:
            print(f'    Batch {i+1}/{len(loader)} | Loss: {epoch_loss/(i+1):.4f}', end='\r')
    return epoch_loss / len(loader)


def evaluate_epoch(model, loader, criterion, device):
    model.eval()
    epoch_loss = 0.0
    with torch.no_grad():
        for src, src_lens, trg in loader:
            src, trg = src.to(device), trg.to(device)
            outputs, _ = model(src, src_lens, trg, teacher_forcing_ratio=0.0)
            outputs = outputs[:, 1:].reshape(-1, outputs.shape[-1])
            trg     = trg[:, 1:].reshape(-1)
            loss = criterion(outputs, trg)
            epoch_loss += loss.item()
    return epoch_loss / len(loader)


In [ ]:
history = {'train_loss': [], 'val_loss': [], 'train_ppl': [], 'val_ppl': []}
best_val_loss = float('inf')
MODEL_PATH = 'best_model.pt'

print(f'Inicio del entrenamiento ({N_EPOCHS} épocas)')
print('=' * 60)

for epoch in range(N_EPOCHS):
    start_time = time.time()
    tf_ratio = max(0.3, 0.9 - epoch * 0.1)  # programa de teacher forcing

    train_loss = train_epoch(model, train_loader, optimizer, criterion, CLIP,
                             device, tf_ratio)
    val_loss   = evaluate_epoch(model, val_loader, criterion, device)
    scheduler.step(val_loss)

    train_ppl = math.exp(train_loss)
    val_ppl   = math.exp(val_loss)

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_ppl'].append(train_ppl)
    history['val_ppl'].append(val_ppl)

    elapsed = time.time() - start_time
    marker = ''
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save({
            'epoch': epoch,
            'model_state': model.state_dict(),
            'optimizer_state': optimizer.state_dict(),
            'val_loss': val_loss,
        }, MODEL_PATH)
        marker = f'[✓ mejor]'

    print(
        f'Época {epoch+1:2d}/{N_EPOCHS} [{elapsed:.0f}s] '
        f'| Train Loss: {train_loss:.4f} PPL: {train_ppl:.2f} '
        f'| Val Loss: {val_loss:.4f} PPL: {val_ppl:.2f} '
        f'| TF: {tf_ratio:.1f} {marker}'
    )

print('=' * 60)
print(f'[✓] Entrenamiento finalizado. Mejor Val Loss: {best_val_loss:.4f}')


In [ ]:
# Visualización de las curvas de entrenamiento
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Curvas de entrenamiento', fontsize=14, fontweight='bold')

epochs_range = range(1, len(history['train_loss']) + 1)

axes[0].plot(epochs_range, history['train_loss'], 'b-o', label='Train Loss', linewidth=2)
axes[0].plot(epochs_range, history['val_loss'],   'r-o', label='Val Loss',   linewidth=2)
axes[0].set_title('Cross-Entropy Loss')
axes[0].set_xlabel('Época')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs_range, history['train_ppl'], 'b-o', label='Train PPL', linewidth=2)
axes[1].plot(epochs_range, history['val_ppl'],   'r-o', label='Val PPL',   linewidth=2)
axes[1].set_title('Perplexity')
axes[1].set_xlabel('Época')
axes[1].set_ylabel('PPL')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150)
plt.show()


## 7. Inferencia y generación de resúmenes

Se carga el mejor checkpoint y se genera un resumen mediante decodificación
*greedy*. Adicionalmente se almacenan los pesos de atención para su
visualización posterior.


In [ ]:
checkpoint = torch.load(MODEL_PATH, map_location=device)
model.load_state_dict(checkpoint['model_state'])
print(f'[✓] Mejor modelo recargado '
      f'(época {checkpoint["epoch"]+1}, Val Loss: {checkpoint["val_loss"]:.4f}).')


def generate_summary(model, article_text, vocab, max_len=100, device=device):
    """Genera un resumen mediante decodificación greedy."""
    model.eval()
    with torch.no_grad():
        src_ids  = vocab.encode(article_text, max_len=MAX_ARTICLE_LEN)
        src      = torch.LongTensor(src_ids).unsqueeze(0).to(device)
        src_lens = torch.LongTensor([len(src_ids)])

        enc_out, hidden, cell = model.encoder(src, src_lens)
        src_mask = model.create_mask(src)

        tokens     = [vocab.sos_idx]
        attn_store = []
        input_tok  = torch.LongTensor([vocab.sos_idx]).to(device)

        for _ in range(max_len):
            pred, hidden, cell, attn_w = model.decoder(
                input_tok, hidden, cell, enc_out, src_mask
            )
            top1 = pred.argmax(1).item()
            tokens.append(top1)
            attn_store.append(attn_w.squeeze(0).cpu().numpy())
            if top1 == vocab.eos_idx:
                break
            input_tok = torch.LongTensor([top1]).to(device)

        return vocab.decode(tokens), attn_store, src_ids


# Ejemplos de generación
print('=' * 60)
print('Ejemplos de resúmenes generados')
print('=' * 60)
for i in range(3):
    sample = test_data[i]
    generated, attn_weights, src_ids = generate_summary(model, sample['article'], vocab)
    print(f'\nEjemplo {i+1}:')
    print(f'  Artículo (200 chars) : {sample["article"][:200]}...')
    print(f'  Referencia            : {sample["highlights"][:200]}')
    print(f'  Generado              : {generated}')


In [ ]:
# Visualización del mapa de atención sobre un ejemplo
def plot_attention(article_text, summary_text, attn_weights, max_src=40, max_trg=20):
    src_tokens = simple_tokenize(article_text)[:max_src]
    trg_tokens = simple_tokenize(summary_text)[:max_trg]
    n_trg = min(len(attn_weights), len(trg_tokens))
    n_src = min(len(src_tokens), attn_weights[0].shape[0]) if attn_weights else max_src
    attn_matrix = np.array([attn_weights[t][:n_src] for t in range(n_trg)])

    fig, ax = plt.subplots(figsize=(max(8, n_src // 2), max(4, n_trg // 2 + 2)))
    im = ax.imshow(attn_matrix, cmap='viridis', aspect='auto')
    ax.set_xticks(range(n_src))
    ax.set_yticks(range(n_trg))
    ax.set_xticklabels(src_tokens[:n_src], rotation=90, fontsize=8)
    ax.set_yticklabels(trg_tokens[:n_trg], fontsize=9)
    ax.set_xlabel('Artículo (tokens fuente)')
    ax.set_ylabel('Resumen (tokens objetivo)')
    ax.set_title('Mapa de atención (Bahdanau)', fontsize=13, fontweight='bold')
    plt.colorbar(im, ax=ax)
    plt.tight_layout()
    plt.savefig('attention_map.png', dpi=150, bbox_inches='tight')
    plt.show()


sample = test_data[0]
generated, attn_weights, src_ids = generate_summary(model, sample['article'], vocab)
plot_attention(sample['article'], generated, attn_weights)


## 8. Evaluación con métricas ROUGE

Se utilizan las métricas **ROUGE-1**, **ROUGE-2** y **ROUGE-L** (Lin, 2004),
que cuantifican el solapamiento de unigramas, bigramas y la subsecuencia común
más larga entre el resumen generado y el de referencia. Se reporta la
F-measure promedio sobre una submuestra del conjunto de prueba.


In [ ]:
def compute_rouge(model, dataset, vocab, n_samples=500):
    """ROUGE-1, ROUGE-2 y ROUGE-L (F-measure) sobre n_samples ejemplos."""
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
    scores = {'rouge1': [], 'rouge2': [], 'rougeL': []}
    model.eval()
    for i in range(min(n_samples, len(dataset))):
        sample = dataset[i]
        generated, _, _ = generate_summary(model, sample['article'], vocab, max_len=80)
        result = scorer.score(sample['highlights'], generated)
        for k in scores:
            scores[k].append(result[k].fmeasure)
        if (i + 1) % 50 == 0:
            print(f'    Procesado {i+1}/{n_samples} ...', end='\r')
    return {k: float(np.mean(v)) for k, v in scores.items()}


print(f'Calculando ROUGE sobre {ROUGE_SAMPLES} muestras del conjunto de prueba ...')
rouge_scores = compute_rouge(model, test_data, vocab, n_samples=ROUGE_SAMPLES)

print('\nResultados ROUGE:')
print('=' * 40)
for metric, score in rouge_scores.items():
    print(f'    {metric.upper():<8}: {score:.4f} ({score*100:.2f}%)')

print('\nComparación con la literatura:')
baselines = [
    ('Lead-3 (extractivo)',   0.401, 0.175, 0.365),
    ('Seq2Seq básico',        0.358, 0.144, 0.330),
    ('Seq2Seq + Atención',    0.374, 0.158, 0.346),
    ('BART (SOTA)',           0.448, 0.214, 0.412),
    ('Modelo propuesto',      rouge_scores['rouge1'], rouge_scores['rouge2'], rouge_scores['rougeL']),
]
print(f"    {'Modelo':<28} {'R-1':>6} {'R-2':>6} {'R-L':>6}")
print('    ' + '-' * 50)
for name, r1, r2, rl in baselines:
    print(f"    {name:<28} {r1:.3f}  {r2:.3f}  {rl:.3f}")


In [ ]:
# Visualización comparativa de métricas ROUGE
metrics = ['ROUGE-1', 'ROUGE-2', 'ROUGE-L']
systems = ['Lead-3', 'Seq2Seq', 'Seq2Seq\n+Atención', 'Modelo\npropuesto', 'BART']
scores_matrix = [
    [0.401, 0.175, 0.365],
    [0.358, 0.144, 0.330],
    [0.374, 0.158, 0.346],
    [rouge_scores['rouge1'], rouge_scores['rouge2'], rouge_scores['rougeL']],
    [0.448, 0.214, 0.412],
]

x = np.arange(len(systems))
width = 0.25
colors = ['#2196F3', '#FF9800', '#4CAF50']

fig, ax = plt.subplots(figsize=(12, 6))
for i, (metric, color) in enumerate(zip(metrics, colors)):
    vals = [s[i] for s in scores_matrix]
    bars = ax.bar(x + i*width, vals, width, label=metric, color=color,
                  alpha=0.85, edgecolor='white')
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f'{v:.3f}', ha='center', va='bottom', fontsize=8)

ax.set_xticks(x + width)
ax.set_xticklabels(systems)
ax.set_ylim(0, 0.55)
ax.set_ylabel('Score ROUGE (F-measure)')
ax.set_title('Comparación de métricas ROUGE — CNN/DailyMail',
             fontsize=13, fontweight='bold')
ax.legend(loc='upper left')
ax.grid(axis='y', alpha=0.3)
ax.axvspan(2.6, 3.4, alpha=0.1, color='gold')

plt.tight_layout()
plt.savefig('rouge_comparison.png', dpi=150)
plt.show()


## 9. Persistencia de artefactos

Se guardan los pesos del mejor modelo, el vocabulario y la configuración para
permitir su consumo por la aplicación Streamlit asociada al proyecto.


In [ ]:
with open('vocab.pkl', 'wb') as f:
    pickle.dump(vocab, f)

model_config = {
    'vocab_size'      : len(vocab),
    'embed_dim'       : EMBED_DIM,
    'hidden_dim'      : HIDDEN_DIM,
    'n_layers'        : N_LAYERS,
    'dropout'         : DROPOUT,
    'max_article_len' : MAX_ARTICLE_LEN,
    'max_summary_len' : MAX_SUMMARY_LEN,
    'rouge_scores'    : rouge_scores,
}
with open('model_config.pkl', 'wb') as f:
    pickle.dump(model_config, f)

print(f'[✓] Artefactos guardados:')
for fname in ['best_model.pt', 'vocab.pkl', 'model_config.pkl',
              'training_curves.png', 'attention_map.png', 'rouge_comparison.png']:
    print(f'    - {fname}')


## 10. Interfaz interactiva: resumen sobre documentos arbitrarios

Las celdas siguientes implementan una interfaz interactiva basada en
`ipywidgets` que permite probar el modelo entrenado sobre documentos
arbitrarios sin salir del notebook. Soporta tres canales de entrada:

1. Cargue de archivos **PDF**, **EPUB** o **TXT** mediante `FileUpload`.
2. Pegado directo de texto en un `Textarea`.

El flujo replica el comportamiento de la aplicación Streamlit asociada al
proyecto: detección de idioma, traducción ES → EN del documento (si aplica),
generación del resumen con el modelo Encoder-Decoder entrenado y traducción
EN → ES del resumen final cuando el usuario solicita la salida en español.

Las traducciones utilizan los modelos `Helsinki-NLP/opus-mt-es-en` y
`Helsinki-NLP/opus-mt-en-es` (MarianMT), que se descargan bajo demanda desde
Hugging Face Hub.


In [ ]:
# Instalación de dependencias adicionales para la interfaz interactiva
!pip install -q ipywidgets pypdf ebooklib beautifulsoup4 lxml


In [ ]:
# Imports y utilidades para la interfaz
import io
import os
import re
import tempfile

import ipywidgets as widgets
from IPython.display import display, HTML, clear_output


# ───────── Extracción de texto desde archivos ─────────
def extract_text_from_pdf(file_bytes):
    """Extrae texto plano de un PDF mediante pypdf."""
    try:
        from pypdf import PdfReader
    except ImportError:
        from PyPDF2 import PdfReader
    reader = PdfReader(io.BytesIO(file_bytes))
    chunks = []
    for page in reader.pages:
        try:
            chunks.append(page.extract_text() or "")
        except Exception:
            continue
    return "\n".join(chunks).strip()


def extract_text_from_epub(file_bytes):
    """Extrae texto plano de un EPUB con ebooklib + BeautifulSoup."""
    from ebooklib import epub, ITEM_DOCUMENT
    from bs4 import BeautifulSoup

    with tempfile.NamedTemporaryFile(suffix=".epub", delete=False) as tmp:
        tmp.write(file_bytes)
        tmp_path = tmp.name
    try:
        book = epub.read_epub(tmp_path)
        chunks = []
        for item in book.get_items_of_type(ITEM_DOCUMENT):
            soup = BeautifulSoup(item.get_content(), "html.parser")
            chunks.append(soup.get_text(separator=" ", strip=True))
    finally:
        try:
            os.remove(tmp_path)
        except OSError:
            pass
    return "\n".join(chunks).strip()


def decode_txt(file_bytes):
    """Decodifica un TXT con tolerancia a varias codificaciones."""
    for enc in ("utf-8", "utf-8-sig", "latin-1", "cp1252"):
        try:
            return file_bytes.decode(enc)
        except UnicodeDecodeError:
            continue
    return file_bytes.decode("utf-8", errors="replace")


def extract_uploaded_file(filename, content_bytes):
    """Despacha la extracción de texto según la extensión del archivo."""
    name = filename.lower()
    if name.endswith(".pdf"):
        return extract_text_from_pdf(content_bytes)
    if name.endswith(".epub"):
        return extract_text_from_epub(content_bytes)
    if name.endswith(".txt"):
        return decode_txt(content_bytes)
    raise ValueError(f"Formato no soportado: {filename}")


print(f'[✓] Funciones de extracción de archivos definidas.')


In [ ]:
# ───────── Detección heurística de idioma ─────────
SPANISH_HINTS = {
    "que", "para", "como", "pero", "más", "mas", "donde", "cuando",
    "porque", "esta", "está", "estaba", "fue", "fueron", "uno", "una",
    "del", "los", "las", "señor", "señora", "años", "día", "noche",
    "casa", "tiempo", "hombre", "mujer", "muy", "también", "sólo", "solo",
    "sino", "sin", "según", "después", "antes", "entonces",
}
ENGLISH_HINTS = {
    "the", "and", "of", "to", "in", "is", "was", "were", "are", "for",
    "with", "that", "this", "from", "have", "has", "had", "their", "they",
    "would", "could", "should", "where", "when", "while", "after", "before",
    "according", "between", "through", "however", "because",
}


def detect_language(text):
    sample = text[:4000].lower()
    tokens = re.findall(r"[a-záéíóúñü]+", sample, flags=re.IGNORECASE)
    if len(tokens) < 5:
        return "unknown"
    es_hits = sum(1 for t in tokens if t in SPANISH_HINTS)
    en_hits = sum(1 for t in tokens if t in ENGLISH_HINTS)
    has_spanish_chars = bool(re.search(r"[áéíóúñ¡¿]", sample))
    if has_spanish_chars and es_hits >= en_hits:
        return "es"
    if es_hits > en_hits * 1.2:
        return "es"
    if en_hits > es_hits * 1.2:
        return "en"
    return "unknown"


# ───────── Traductores MarianMT vía AutoTokenizer + AutoModel ─────────
# Se cargan con AutoModelForSeq2SeqLM en lugar de transformers.pipeline()
# para evitar errores como "Invalid translation task translation, use
# 'translation_XX_to_YY' format" que aparecen con versiones recientes
# de transformers donde el registro de tareas del pipeline ha cambiado.

_translator_es_en = None
_translator_en_es = None


def _load_seq2seq(model_name):
    """Devuelve la tupla (tokenizer, model) descargada de Hugging Face."""
    from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
    print(f'   Descargando {model_name} (primera ejecución) ...')
    tok = AutoTokenizer.from_pretrained(model_name)
    mdl = AutoModelForSeq2SeqLM.from_pretrained(model_name)
    mdl.eval()
    return tok, mdl


def get_translator_es_en():
    global _translator_es_en
    if _translator_es_en is None:
        _translator_es_en = _load_seq2seq("Helsinki-NLP/opus-mt-es-en")
    return _translator_es_en


def get_translator_en_es():
    global _translator_en_es
    if _translator_en_es is None:
        _translator_en_es = _load_seq2seq("Helsinki-NLP/opus-mt-en-es")
    return _translator_en_es


def _seq2seq_generate(text, tokenizer_model, max_length=512, num_beams=4):
    """Tokenize → generate → decode usando las clases base."""
    tok, mdl = tokenizer_model
    inputs = tok(text, return_tensors="pt", truncation=True, max_length=512)
    with torch.no_grad():
        out_ids = mdl.generate(
            **inputs,
            max_length=max_length,
            num_beams=num_beams,
            early_stopping=True,
        )
    return tok.decode(out_ids[0], skip_special_tokens=True)


def translate_in_chunks(text, tokenizer_model, max_chars=350):
    """Traducción robusta para textos largos: trozos por oración."""
    if not text.strip():
        return ""
    sentences = re.split(r"(?<=[\.\!\?])\s+", text.strip())
    out, buffer = [], ""
    for s in sentences:
        if len(buffer) + len(s) < max_chars:
            buffer = (buffer + " " + s).strip()
        else:
            if buffer:
                out.append(_seq2seq_generate(buffer, tokenizer_model))
            buffer = s
    if buffer:
        out.append(_seq2seq_generate(buffer, tokenizer_model))
    return " ".join(out)


print(f'[✓] Detección de idioma y traductores configurados (AutoModel).')


In [ ]:
# ───────── Construcción de la interfaz con ipywidgets ─────────

# Componentes de entrada
file_uploader = widgets.FileUpload(
    accept='.pdf,.epub,.txt',
    multiple=False,
    description='Cargar archivo',
    layout=widgets.Layout(width='auto'),
)

text_input = widgets.Textarea(
    placeholder=(
        'O pegue aquí el texto a resumir. Si está en español y el toggle '
        'ES → EN está activo, se traducirá automáticamente antes de resumir.'
    ),
    layout=widgets.Layout(width='100%', height='160px'),
)

output_language = widgets.RadioButtons(
    options=['Español', 'Inglés'],
    value='Español',
    description='Idioma del resumen:',
    style={'description_width': 'initial'},
)

auto_translate_input = widgets.Checkbox(
    value=True,
    description='Traducir entrada ES → EN automáticamente',
)

max_len_slider = widgets.IntSlider(
    value=80, min=30, max=200, step=5,
    description='Longitud máxima del resumen (tokens):',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='90%'),
)

generate_btn = widgets.Button(
    description='Generar resumen',
    button_style='primary',
    icon='check',
)

clear_btn = widgets.Button(description='Limpiar')

output_area = widgets.Output()


def get_uploaded_file_content():
    """Obtiene (nombre, bytes) del archivo cargado, soportando ipywidgets v7 y v8."""
    val = file_uploader.value
    if not val:
        return None, None
    if isinstance(val, dict):  # ipywidgets v7
        for fname, info in val.items():
            return fname, info['content']
    if isinstance(val, (tuple, list)):  # ipywidgets v8
        f = val[0]
        return f.get('name'), f.get('content')
    return None, None


def render_summary_card(title, body):
    color = '#10b981' if 'español' in title.lower() else '#6366f1'
    label = '#6ee7b7' if 'español' in title.lower() else '#a5b4fc'
    display(HTML(f"""
    <div style='border-left: 4px solid {color}; padding: 1em 1.2em;
                background: #f1f5f9; border-radius: 8px; margin: 1em 0;'>
      <div style='font-size: 0.75rem; font-weight: 700; text-transform: uppercase;
                  letter-spacing: 0.1em; color: {label}; margin-bottom: 0.5em;'>
        {title}
      </div>
      <div style='color: #1e293b; line-height: 1.6;'>{body}</div>
    </div>
    """))


def on_generate(_btn):
    with output_area:
        clear_output(wait=True)

        # 1) Obtener texto
        fname, content = get_uploaded_file_content()
        if content is not None and fname:
            try:
                text = extract_uploaded_file(fname, content)
                print(f'[✓] {len(text.split()):,} palabras extraídas de {fname}.')
            except Exception as exc:
                print(f'Error al procesar {fname}: {exc}')
                return
        else:
            text = (text_input.value or '').strip()

        if not text or len(text) < 20:
            print('Cargue un archivo o pegue al menos 20 caracteres de texto.')
            return

        # 2) Detección de idioma y traducción de entrada si aplica
        lang = detect_language(text)
        text_for_model = text
        if lang == 'es' and auto_translate_input.value:
            print('Idioma detectado: español. Traduciendo a inglés ...')
            try:
                text_for_model = translate_in_chunks(text[:12000], get_translator_es_en())
                print(f'[✓] Traducción completada ({len(text_for_model.split()):,} palabras).')
            except Exception as exc:
                print(f'Falló la traducción ES → EN: {exc}. Continuando con texto original.')
        elif lang == 'es' and not auto_translate_input.value:
            print('Aviso: el texto parece estar en español y la traducción automática '
                  'está desactivada. El modelo está entrenado en inglés; los resultados '
                  'pueden ser pobres.')

        # 3) Generar resumen con el modelo Encoder-Decoder
        print('Generando resumen ...')
        summary_en, _, _ = generate_summary(
            model, text_for_model, vocab, max_len=max_len_slider.value
        )

        # 4) Traducir al idioma de salida si aplica
        if output_language.value == 'Español':
            try:
                summary_out = translate_in_chunks(summary_en, get_translator_en_es())
            except Exception as exc:
                print(f'Falló la traducción EN → ES: {exc}. Mostrando resumen en inglés.')
                summary_out = summary_en
                render_summary_card('Resumen (inglés)', summary_out)
                return
            render_summary_card('Resumen (español)', summary_out)
            print('Versión original generada por el modelo (inglés):')
            print(f'   {summary_en}')
        else:
            render_summary_card('Resumen (inglés)', summary_en)


def on_clear(_btn):
    with output_area:
        clear_output()
    file_uploader.value = ()
    text_input.value = ''


generate_btn.on_click(on_generate)
clear_btn.on_click(on_clear)


# Composición del layout
ui = widgets.VBox([
    widgets.HTML('<h4 style="margin: 0;">Cargue de documento (PDF, EPUB o TXT)</h4>'),
    file_uploader,
    widgets.HTML('<p style="margin-top: 1em;">O pegue el texto a resumir:</p>'),
    text_input,
    widgets.HBox([output_language, auto_translate_input]),
    max_len_slider,
    widgets.HBox([generate_btn, clear_btn]),
    output_area,
])

display(ui)


## 11. Análisis y conclusiones

### 11.1 Hallazgos principales

1. **Mecanismo de atención.** Los mapas de atención evidencian que el decoder
   concentra su masa de probabilidad sobre las posiciones del artículo
   semánticamente relevantes para cada token generado, validando la hipótesis
   de Bahdanau et al. (2015) sobre la utilidad del alineamiento *soft*.
2. **Embeddings preentrenados.** La inicialización con GloVe acelera la
   convergencia y mejora la cobertura léxica frente a embeddings aleatorios,
   especialmente para términos de baja frecuencia.
3. **Programa de teacher forcing.** Disminuir gradualmente el ratio de
   *teacher forcing* reduce la brecha entre el régimen de entrenamiento y el
   de inferencia (*exposure bias*), mejorando la robustez en generación libre.

### 11.2 Limitaciones

El modelo Encoder-Decoder con LSTM presenta limitaciones reconocidas frente a
las arquitecturas Transformer (Vaswani et al., 2017): paralelización limitada,
dificultad para capturar dependencias de largo alcance y rendimiento ROUGE
inferior al de modelos preentrenados como BART (Lewis et al., 2020) o PEGASUS
(Zhang et al., 2020). El mecanismo *pointer-generator* (See et al., 2017)
también aliviaría el problema de palabras fuera de vocabulario.

### 11.3 Trabajo futuro

Las extensiones naturales del trabajo incluyen: (i) reemplazar la arquitectura
recurrente por un Transformer, (ii) incorporar un mecanismo *coverage* para
reducir repeticiones, (iii) habilitar un módulo *copy* para nombres propios y
entidades fuera de vocabulario, y (iv) escalar el entrenamiento al corpus
completo.

### 11.4 Referencias

- Bahdanau, D., Cho, K. & Bengio, Y. (2015). *Neural Machine Translation by
  Jointly Learning to Align and Translate*. ICLR.
- Hermann, K. M. *et al.* (2015). *Teaching Machines to Read and Comprehend*.
  NeurIPS.
- Lewis, M. *et al.* (2020). *BART: Denoising Sequence-to-Sequence Pre-training*.
  ACL.
- Lin, C.-Y. (2004). *ROUGE: A Package for Automatic Evaluation of Summaries*.
  ACL Workshop.
- Pennington, J., Socher, R. & Manning, C. (2014). *GloVe: Global Vectors for
  Word Representation*. EMNLP.
- See, A., Liu, P. J. & Manning, C. D. (2017). *Get to the Point: Summarization
  with Pointer-Generator Networks*. ACL.
- Vaswani, A. *et al.* (2017). *Attention is All You Need*. NeurIPS.
- Zhang, J. *et al.* (2020). *PEGASUS: Pre-training with Extracted Gap-sentences
  for Abstractive Summarization*. ICML.
